# Large Language Models on HPC Clusters

## Introduction to LLMs and HPC Infrastructure

This workshop introduces you to running Large Language Models (LLMs) on High Performance Computing (HPC) clusters. You'll learn how to leverage GPU resources (L4, A100, H100) for LLM inference, fine-tuning, and production deployment.

### Learning Objectives

By the end of this workshop, you will be able to:

* Understand LLM architectures and computational requirements
* Set up and configure LLM environments on HPC clusters
* Run local LLM inference on L4 GPUs
* Fine-tune models using distributed training
* Deploy LLM services using Slurm job scheduling
* Optimize performance and resource utilization

### Workshop Structure (3-4 hours)

1. **Introduction** (30 min) - LLM basics and HPC setup
2. **Local LLM Inference** (60 min) - Running models on L4 GPUs
3. **Fine-tuning** (90 min) - Training and adapting models
4. **Production Deployment** (60 min) - Slurm-based LLM services

### Prerequisites

* Basic Python programming
* Familiarity with Jupyter notebooks
* Basic understanding of machine learning concepts
* Access to HPC cluster with GPU resources

## Understanding Large Language Models

Large Language Models are transformer-based neural networks trained on massive text datasets. They can generate human-like text, answer questions, and perform various language tasks.

### Key Characteristics:

* **Size**: Billions to trillions of parameters
* **Memory**: Requires significant GPU memory (8GB+ for inference)
* **Compute**: Intensive matrix operations benefit from GPU acceleration
* **Storage**: Large model files (several GB to TB)

### Popular LLM Families:

* **GPT** (OpenAI): Generative Pre-trained Transformer
* **LLaMA** (Meta): Open-source, efficient architecture
* **Mistral**: High-performance open models
* **Code Llama**: Specialized for code generation
* **Falcon**: Open-source models from Technology Innovation Institute

## HPC GPU Resources

Our cluster provides three types of GPUs optimized for different workloads:

### L4 GPUs (Recommended for this workshop)
* **Memory**: 24GB VRAM
* **Use case**: Inference, small fine-tuning
* **Availability**: High (most accessible)
* **Power**: Efficient for most LLM tasks

### A100 GPUs
* **Memory**: 40GB or 80GB VRAM
* **Use case**: Large model training, multi-GPU setups
* **Availability**: Medium
* **Power**: High-end training workloads

### H100 GPUs
* **Memory**: 80GB VRAM
* **Use case**: Cutting-edge research, largest models
* **Availability**: Limited
* **Power**: Latest generation, highest performance

## Environment Setup

Let's set up our Python environment with the necessary libraries for LLM work.

In [ ]:
import torch
import transformers
import accelerate
import bitsandbytes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import psutil
import GPUtil
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## GPU Detection and Information

Let's check what GPU resources are available and their specifications.

In [ ]:
def get_gpu_info():
    """Get detailed GPU information"""
    if torch.cuda.is_available():
        print("🎯 GPU Information:")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
        print(f"Current GPU: {torch.cuda.current_device()}")
        
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"\nGPU {i}:")
            print(f"  Name: {props.name}")
            print(f"  Memory: {props.total_memory / 1024**3:.1f} GB")
            print(f"  Compute Capability: {props.major}.{props.minor}")
            print(f"  Multiprocessors: {props.multi_processor_count}")
            
            # Check memory usage
            memory_allocated = torch.cuda.memory_allocated(i) / 1024**3
            memory_reserved = torch.cuda.memory_reserved(i) / 1024**3
            print(f"  Memory Allocated: {memory_allocated:.2f} GB")
            print(f"  Memory Reserved: {memory_reserved:.2f} GB")
    else:
        print("❌ No CUDA GPUs available")
        print("Make sure you're running on a GPU node!")

get_gpu_info()

## System Resource Monitoring

Let's also check our system resources to understand the compute environment.

In [ ]:
def get_system_info():
    """Get system resource information"""
    print("🖥️ System Information:")
    print(f"CPU: {psutil.cpu_count()} cores")
    print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
    print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")
    
    # Check if we're in a Slurm job
    slurm_job_id = os.environ.get('SLURM_JOB_ID')
    if slurm_job_id:
        print(f"\n📋 Slurm Job Information:")
        print(f"Job ID: {slurm_job_id}")
        print(f"Node: {os.environ.get('SLURM_NODELIST', 'Unknown')}")
        print(f"Partition: {os.environ.get('SLURM_JOB_PARTITION', 'Unknown')}")
        print(f"Time Limit: {os.environ.get('SLURM_TIME_LIMIT', 'Unknown')}")
    else:
        print("\n💡 Running interactively (not in Slurm job)")

get_system_info()

## LLM Model Sizes and Memory Requirements

Understanding model sizes helps us choose appropriate models for our GPU resources.

In [ ]:
# Common LLM model sizes and memory requirements
model_info = {
    'Model': ['GPT-2 (117M)', 'GPT-2 (345M)', 'GPT-2 (774M)', 'LLaMA-7B', 'LLaMA-13B', 'LLaMA-30B', 'LLaMA-65B'],
    'Parameters': ['117M', '345M', '774M', '7B', '13B', '30B', '65B'],
    'Model Size (GB)': [0.5, 1.4, 3.1, 13, 26, 60, 130],
    'Inference RAM (GB)': [1, 2, 4, 14, 26, 60, 130],
    'Training RAM (GB)': [2, 4, 8, 28, 52, 120, 260],
    'Recommended GPU': ['L4', 'L4', 'L4', 'L4', 'A100', 'A100', 'H100']
}

df = pd.DataFrame(model_info)
print("📊 LLM Model Size and Memory Requirements:")
print(df.to_string(index=False))

# Visualize memory requirements
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.bar(df['Model'], df['Model Size (GB)'])
plt.title('Model Size (GB)')
plt.xticks(rotation=45)
plt.ylabel('Size (GB)')

plt.subplot(2, 2, 2)
plt.bar(df['Model'], df['Inference RAM (GB)'])
plt.title('Inference Memory (GB)')
plt.xticks(rotation=45)
plt.ylabel('RAM (GB)')

plt.subplot(2, 2, 3)
plt.bar(df['Model'], df['Training RAM (GB)'])
plt.title('Training Memory (GB)')
plt.xticks(rotation=45)
plt.ylabel('RAM (GB)')

plt.subplot(2, 2, 4)
gpu_counts = df['Recommended GPU'].value_counts()
plt.pie(gpu_counts.values, labels=gpu_counts.index, autopct='%1.1f%%')
plt.title('GPU Distribution')

plt.tight_layout()
plt.show()

## Workshop Overview

### What We'll Cover:

1. **Notebook 2: Local LLM Inference**
   - Loading and running models on L4 GPUs
   - Memory optimization techniques
   - Batch processing and throughput optimization

2. **Notebook 3: Fine-tuning**
   - Preparing datasets for fine-tuning
   - LoRA (Low-Rank Adaptation) for efficient training
   - Distributed training across multiple GPUs

3. **Notebook 4: Production Deployment**
   - Creating Slurm job scripts
   - Building LLM services and APIs
   - Monitoring and scaling

### Best Practices for HPC LLM Work:

* **Resource Management**: Always check GPU memory usage
* **Model Selection**: Choose models appropriate for your GPU memory
* **Batch Processing**: Process multiple requests together when possible
* **Caching**: Cache models and tokenizers to avoid reloading
* **Monitoring**: Track resource usage and performance metrics
* **Cleanup**: Always clear GPU memory when done

## Next Steps

Now that we understand the basics, let's move to the next notebook where we'll:

1. Load and run our first LLM on L4 GPUs
2. Explore different model architectures
3. Optimize memory usage and performance
4. Build a simple chat interface

**Ready to proceed?** Let's start with local LLM inference!

---
**Next notebook:** [Local LLM Inference on L4 GPUs](02_local_llm.ipynb)

// ...at the top of each notebook...
{
 "cell_type": "markdown",
 "metadata": {},
 "source": [
  "© mattbixley"
 ]
},